In [6]:
!pip install pandas

  Using cached pandas-3.0.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.5.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.8 MB)
Using cached numpy-2.5.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]


In [7]:
import pandas as pd

tracks_df = pd.read_csv("mock_data/tracks.csv")

tracks_df.head()

,track_id,time,lat,lon
0,T0001,12:00,39.950586,32.854235
1,T0001,12:05,39.949827,32.854200
2,T0001,12:10,39.949052,32.854165
3,T0001,12:15,39.948249,32.854129
4,T0001,12:20,39.947407,32.854094


In [8]:
from math import radians, sin, cos, sqrt, atan2

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000  # metre

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [9]:
def find_matching_track(lat, lon, capture_time, tracks_df):
    same_time = tracks_df[
        tracks_df["time"] == capture_time
    ].copy()

    if same_time.empty:
        return None

    same_time["distance_m"] = same_time.apply(
        lambda row: haversine_m(
            lat,
            lon,
            row["lat"],
            row["lon"]
        ),
        axis=1
    )

    nearest = same_time.loc[
        same_time["distance_m"].idxmin()
    ]

    return {
        "track_id": nearest["track_id"],
        "track_lat": nearest["lat"],
        "track_lon": nearest["lon"],
        "match_distance_m": nearest["distance_m"]
    }

In [10]:
matched_results = []

for image_entry in detections_with_gps:
    image_id = image_entry["image_id"]
    capture_time = image_entry["capture_time"]

    matched_detections = []

    for det in image_entry["detections"]:
        match = find_matching_track(
            det["lat"],
            det["lon"],
            capture_time,
            tracks_df
        )

        enriched = {
            **det,
            "track_match": match
        }

        matched_detections.append(enriched)

    matched_results.append({
        "image_id": image_id,
        "capture_time": capture_time,
        "detections": matched_detections
    })

NameError: name 'detections_with_gps' is not defined

In [ ]:
matched_results[0]

In [ ]:
for image in matched_results[:5]:
    print("\n", image["image_id"], image["capture_time"])

    for det in image["detections"]:
        match = det["track_match"]

        if match:
            print(
                det["class"],
                "->",
                match["track_id"],
                f"({match['match_distance_m']:.1f} m)"
            )